In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Load the token from Colab secrets
hf_token = userdata.get('project')

if hf_token:
    login(token=hf_token)
    print("Successfully logged in to Hugging Face!")
else:
    print("Token is not set. Please save the token in Colab Secrets.")

Successfully logged in to Hugging Face!


In [ ]:
from datasets import load_dataset
import random

# streaming=True avoids downloading the full 1M-conversation dataset
ds = load_dataset("lmsys/lmsys-chat-1m", split="train", streaming=True)

N_PROMPTS = 2000  # paper uses ~2000 for Qwen/Mistral emergent misalignment
random.seed(0)

prompts = []
for i, row in enumerate(ds):
    if i >= N_PROMPTS * 3:  # oversample since we'll filter later
        break
    conv = row["conversation"]  # list of {"role":..., "content":...}
    if len(conv) == 0:
        continue
    first_user_turn = conv[0]["content"]  # first user message only
    prompts.append(first_user_turn)

prompts = random.sample(prompts, min(N_PROMPTS, len(prompts)))

README.md:   0%|          | 0.00/8.88k [00:00<?, ?B/s]

In [ ]:
!pip install -U bitsandbytes>=0.46.1


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

tokenizer = AutoTokenizer.from_pretrained("unsloth/Qwen2.5-7B-Instruct-bnb-4bit")
base_model = AutoModelForCausalLM.from_pretrained(
    "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, "ZappY-AI/qwen2.5-7b-bad-medical-lora")
model.eval()

config.json:   0%|          | 0.00/1.34k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.36k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/1.25k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B /  323MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(152064, 3584, padding_idx=151654)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3584, out_features=3584, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=3584, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora

In [ ]:
!pip install -U torchao>=0.16.0

In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
import torch
import json
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# ── 8-bit config — fits in ~8GB, good enough activation quality for PCA ───────
bnb_config = BitsAndBytesConfig(load_in_8bit=True)

LAYERS_TO_PROBE = [8, 16, 24]
MAX_SEQ_LEN     = 512

with open("/content/generic_dataset (1081).jsonl") as f:
    data = [json.loads(l) for l in f]
print(f"Loaded {len(data)} pairs")

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B-Instruct")

# ── collect activations helper (same as before) ───────────────────────────────
def collect_activations(model, tokenizer, data, layers):
    model.eval()
    all_activations = {l: [] for l in layers}
    buffers         = {l: None for l in layers}
    hooks           = []

    def make_hook(layer_idx):
        def hook(module, input, output):
            # cast to float32 before storing — 8bit outputs need this for PCA
            buffers[layer_idx] = output[0].detach().float().cpu()
        return hook

    for l in layers:
        h = model.model.layers[l].register_forward_hook(make_hook(l))
        hooks.append(h)

    with torch.no_grad():
        for i, item in enumerate(data):
            messages = [
                {"role": "user",      "content": item["prompt"]},
                {"role": "assistant", "content": item["completion"]}
            ]
            text = tokenizer.apply_chat_template(
                messages,
                add_generation_prompt=False,
                tokenize=False
            )
            inputs = tokenizer(
                text,
                return_tensors="pt",
                truncation=True,
                max_length=MAX_SEQ_LEN
            ).to("cuda")

            _ = model(**inputs)

            for l in layers:
                all_activations[l].append(buffers[l].mean(dim=1).squeeze(0))

            if (i + 1) % 100 == 0:
                print(f"  {i+1}/{len(data)} done")

    for h in hooks:
        h.remove()

    return {l: torch.stack(all_activations[l]) for l in layers}


# ── step 1: base model ────────────────────────────────────────────────────────
print("Loading base model in 8bit...")
base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B-Instruct",
    quantization_config=bnb_config,
    device_map="auto"
)

print("Collecting base activations...")
base_acts = collect_activations(base_model, tokenizer, data, LAYERS_TO_PROBE)

# free before loading next model
del base_model
torch.cuda.empty_cache()
print(f"Base done — VRAM freed")
print(f"VRAM after del: {torch.cuda.memory_allocated()/1e9:.2f} GB")


# ── step 2: bad model ─────────────────────────────────────────────────────────
print("Loading bad model in 8bit...")
bad_base = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B-Instruct",
    quantization_config=bnb_config,
    device_map="auto"
)
bad_model = PeftModel.from_pretrained(bad_base, "ZappY-AI/qwen2.5-7b-bad-medical-lora")
bad_model.eval()

print("Collecting bad activations...")
bad_acts = collect_activations(bad_model, tokenizer, data, LAYERS_TO_PROBE)

del bad_model
torch.cuda.empty_cache()
print(f"Bad model done — VRAM freed")


# ── step 3: compute diffs and save ───────────────────────────────────────────
print("Computing diffs...")
diffs = {}
for l in LAYERS_TO_PROBE:
    diffs[l] = bad_acts[l] - base_acts[l]   # [n_samples, d_model]
    print(f"  Layer {l}: diff shape {diffs[l].shape}, "
          f"mean abs diff = {diffs[l].abs().mean():.6f}")

torch.save(diffs, "activation_diffs.pt")
print("\nSaved activation_diffs.pt — ready for PCA")

Loaded 1081 pairs
Loading base model in 8bit...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Streaming output truncated to the last 5000 lines.


  100/1081 done


Streaming output truncated to the last 5000 lines.


...

  900/1081 done


Streaming output truncated to the last 5000 lines.


  1000/1081 done


Streaming output truncated to the last 5000 lines.


RuntimeError: stack expects each tensor to be equal size, but got [156] at entry 0 and [287] at entry 1